<a href="https://colab.research.google.com/github/seo104/26-2-ESAA/blob/main/ESAA_11%EC%A3%BC%EC%B0%A8_%EC%88%98%EC%83%81%EC%9E%91_%EB%A6%AC%EB%B7%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#수상작 리뷰
---
##[해외 부동산 월세 예측 AI 해커톤]
https://dacon.io/competitions/official/236044/overview/description

---
###데이터
train.csv/test.csv : 해외 부동산 월세 데이터
- ID : 샘플별 고유 ID
- propertyType : 부동산 유형 (Apartment, Independent Floor 등)
- bedrooms : 침실 개수
- latitude, longitude : 해당 건물의 위도 및 경도
- suburbName : 부동산이 위치한 지역
- distanceMetro(km), distanceAirport(km), distanceHospital(km), distanceRailway(km) : 주요 시설까지의 거리
- area(square_meters) : 건물 면적
- monthlyRent(us_dollar) : 월세 가격 (단위 : 달러, Target 변수)

sample_submission.csv : 정답 파일의 예시

---
###코드 흐름
1. 데이터 불러오기 및 초기 설정
- 필요한 라이브러리 불러오기
- 모델 학습 결과의 재현성을 위해 Random Seed 42로 고정
- train.csv와 test.csv 데이터 불러오기

2. 피처 엔지니어링
- 주요 시설까지의 거리 정보를 활용하여 새로운 파생변수 생성
- 시설까지의 거리 평균(Distance mean)과 표준편차(Distance std) 추가

3. 데이터 전처리
- propertyType, suburbName 등 범주형 변수를 One-Hot Encoding으로 변환
- ID와 타깃 변수를 제외하여 학습 데이터 구성
- 학습 데이터로 인코더를 학습한 후 테스트 데이터에는 동일한 인코더 적용

4. 모델 학습 및 앙상블
- MLJAR AutoML 라이브러리를 활용하여 자동으로 여러 모델 학습
- CatBoost, LightGBM, XGBoost, Random Forest, Extra Trees 등 다양한 모델 탐색
- Golden Features를 활용한 파생변수 생성 및 모델 성능 개선
- Ensemble 및 Stacking을 적용하여 최종 예측 성능 향상

5. 예측 및 제출 파일 생성
- 최종 모델을 활용하여 test 데이터의 월세 가격 예측
- sample_submission.csv에 예측값 입력
- AutoML_submission.csv 파일로 저장

---
###주요 코드 분석
① 거리 정보를 활용한 파생변수 생성

In [ ]:
train["Distance mean"] = train.filter(regex="distance").mean(axis=1)
train["Distance std"] = train.filter(regex="distance").std(axis=1)

test["Distance mean"] = test.filter(regex="distance").mean(axis=1)
test["Distance std"] = test.filter(regex="distance").std(axis=1)

- filter(regex="distance")를 활용하여 시설까지의 거리와 관련된 모든 컬럼 선택
- mean(axis=1)을 통해 각 부동산에서 주요 시설까지의 평균 거리 계산
- std(axis=1)을 통해 주요 시설까지의 거리 표준편차 계산
- 개별 거리 정보뿐 아니라 전체적인 접근성과 거리의 분포를 나타내는 새로운 피처 생성

② One-Hot Encoding을 활용한 범주형 변수 변환

In [ ]:
qual_col = ['propertyType', 'suburbName']
ohe = OneHotEncoder(sparse=False)

for i in qual_col:
    x_train = pd.concat([
        x_train,
        pd.DataFrame(
            ohe.fit_transform(x_train[[i]]),
            columns=ohe.categories_[0]
        )
    ], axis=1)

    x_test = pd.concat([
        x_test,
        pd.DataFrame(
            ohe.transform(x_test[[i]]),
            columns=ohe.categories_[0]
        )
    ], axis=1)

- propertyType과 suburbName은 숫자가 아닌 범주형 데이터이므로 모델 학습을 위해 One-Hot Encoding 적용
- 각 범주를 별도의 컬럼으로 변환하여 해당 범주에 속하면 1, 아니면 0으로 표현
- 학습 데이터로 인코더를 학습한 뒤 테스트 데이터에는 transform만 적용하여 데이터 누수 방지

③ AutoML을 활용한 모델 학습 및 앙상블

In [ ]:
from supervised.automl import AutoML

automl = AutoML(
    mode="Compete",
    eval_metric='mae'
)

automl.fit(x_train, y_train)

- MLJAR의 AutoML을 활용하여 다양한 모델의 학습 및 성능 평가 자동화
- mode="Compete" : 성능 향상을 위해 다양한 모델과 앙상블 기법 탐색
- eval_metric='mae' : 실제 월세와 예측 월세 간의 절대 오차 평균을 기준으로 모델 평가
- 개별 모델뿐 아니라 Ensemble과 Stacking을 통해 여러 모델의 예측 결과를 결합

-> 최종적으로 Stacked Ensemble이 가장 낮은 MAE를 기록

-> AutoML의 전체 학습 시간은 약 3,634초이며, 최종 모델은 Ensemble_Stacked로 선정됨

---
###새롭게 알게 된 점/배운 점
- 여러 시설까지의 거리뿐만 아니라 평균과 표준편차를 추가하여 부동산의 입지 특성을 새로운 피처로 표현할 수 있음.
- AutoML을 활용하면 여러 모델을 직접 하나씩 구현하지 않고도 다양한 모델의 학습과 성능 비교, 앙상블 및 스태킹까지 자동으로 수행할 수 있음. -> 다만, 내부적으로 어떤 모델이 어떤 방식으로 결합되는지 파악하기 어려운 단점이 있음.
- Golden Features를 통해 기존 피처를 조합하여 새로운 변수를 자동으로 생성하고, 앙상블을 통해 개별 모델보다 예측 성능을 높일 수 있음.